In [1]:
import os

from pyspark.sql import SparkSession

In [10]:
try:
    spark.stop()
except:
    pass

local_driver = "/home/spark/.ivy2/jars/com.clickhouse_clickhouse-jdbc-0.6.5.jar"

spark = (
    SparkSession.builder
    .appName("explore_readmission")
    .config("spark.jars", local_driver)
    .config("spark.driver.extraClassPath", local_driver)
    .config("spark.executor.extraClassPath", local_driver)
    .config("spark.jars.packages", "org.apache.httpcomponents.client5:httpclient5:5.3.1")
    .config("spark.jars.ivy", "/home/spark/.ivy2")
    .config("spark.driver.extraJavaOptions", "-Dclickhouse.logger=org.slf4j.helpers.NOPLogger")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

26/10/26 18:04:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [11]:
jdbc_url = f"jdbc:clickhouse://clickhouse:8123/{os.getenv('CLICKHOUSE_DB', 'default')}"

jdbc_options = {
    "url": jdbc_url,
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

In [12]:
query = """
SELECT
    patient_id,
    encounter_id,
    toString(start_at) AS start_at,
    toString(stop_at) AS stop_at,
    age,
    gender,
    race,
    marital_status,
    total_encounters_before,
    inpatient_count_before,
    emergency_count_before,
    avg_duration_before,
    total_conditions_before,
    chronic_conditions_before,
    distinct_conditions_codes_before,
    readmitted_30d
FROM ehr.features_readmission
"""


df = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", f"({query}) AS t")
    .load()
)

In [14]:
df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- start_at: string (nullable = true)
 |-- stop_at: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- total_encounters_before: integer (nullable = true)
 |-- inpatient_count_before: integer (nullable = true)
 |-- emergency_count_before: integer (nullable = true)
 |-- avg_duration_before: float (nullable = true)
 |-- total_conditions_before: integer (nullable = true)
 |-- chronic_conditions_before: integer (nullable = true)
 |-- distinct_conditions_codes_before: integer (nullable = true)
 |-- readmitted_30d: integer (nullable = true)



In [13]:
print(f"Rows: {df.count()}")
df.show(5)

Rows: 5495
+--------------------+--------------------+-------------------+-------------------+---+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+
|          patient_id|        encounter_id|           start_at|            stop_at|age|gender| race|marital_status|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|readmitted_30d|
+--------------------+--------------------+-------------------+-------------------+---+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+
|03670b6a-8ff5-39e...|03670b6a-8ff5-39e...|1970-01-01 00:00:00|1970-01-01 00:

In [15]:
df.groupBy("readmitted_30d").count().show()

+--------------+-----+
|readmitted_30d|count|
+--------------+-----+
|             1|  940|
|             0| 4555|
+--------------+-----+



In [16]:
df.describe().show()

+-------+--------------------+--------------------+-------------------+-------------------+------------------+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+-------------------+
|summary|          patient_id|        encounter_id|           start_at|            stop_at|               age|gender| race|marital_status|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|     readmitted_30d|
+-------+--------------------+--------------------+-------------------+-------------------+------------------+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+-------------------+
|  c

In [22]:
from pyspark.sql.functions import col, isnan, when

df.filter(isnan(col("avg_duration_before"))).count()

74

In [26]:
df = df.withColumn(
    "avg_duration_before",
    when(isnan(col("avg_duration_before")) | col("avg_duration_before").isNull(), 0.0)
    .otherwise(col("avg_duration_before"))
)

In [27]:
df.filter(isnan(col("avg_duration_before"))).count()

0

In [28]:
df.describe("avg_duration_before").show()

+-------+-------------------+
|summary|avg_duration_before|
+-------+-------------------+
|  count|               5495|
|   mean| 1084.1059673084358|
| stddev|  1513.118106133107|
|    min|                0.0|
|    max|     8711.599609375|
+-------+-------------------+



In [31]:
numeric_cols = [
    "age", "total_encounters_before", "inpatient_count_before",
    "emergency_count_before", "avg_duration_before",
    "total_conditions_before", "chronic_conditions_before",
    "distinct_conditions_codes_before"
]

for numeric_col in numeric_cols:
    nan_count = df.filter(isnan(col(numeric_col))).count()
    null_count = df.filter(col(numeric_col).isNull()).count()
    print(f"{numeric_col}: NaN={nan_count}, NULL={null_count}")

age: NaN=0, NULL=0
total_encounters_before: NaN=0, NULL=0
inpatient_count_before: NaN=0, NULL=0
emergency_count_before: NaN=0, NULL=0
avg_duration_before: NaN=0, NULL=0
total_conditions_before: NaN=0, NULL=0
chronic_conditions_before: NaN=0, NULL=0
distinct_conditions_codes_before: NaN=0, NULL=0


In [32]:
df.describe(numeric_cols).show()

+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|summary|               age|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|
+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|  count|              5495|                   5495|                  5495|                  5495|               5495|                   5495|                     5495|                            5495|
|   mean|56.058052775250225|      74.21747042766151|    7.4787989080982715|     3.164331210191083| 1084.1059673084358|     38.013648771610555|       22.593994540491355|              21.7878070

In [39]:
patient_ids = df.select(col("patient_id")).distinct()

train_ids, test_ids = patient_ids.randomSplit([0.6, 0.4], seed=42)

In [40]:
train_df = df.join(train_ids, on="patient_id", how="inner")
test_df = df.join(test_ids, on="patient_id", how="inner")

In [42]:
print(f"Rows (train): {train_df.count()}, rows (test): {test_df.count()}")

Rows (train): 3396, rows (test): 2099


In [47]:
train_df = train_df.withColumn(
    "gender_encoded",
    when(col("gender") == "M", 1).otherwise(0)
)

In [49]:
train_df.groupBy(col("gender"), col("gender_encoded")).count().show()

+------+--------------+-----+
|gender|gender_encoded|count|
+------+--------------+-----+
|     M|             1| 2108|
|     F|             0| 1288|
+------+--------------+-----+



In [50]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

indexer = StringIndexer(inputCol="marital_status", outputCol="marital_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="marital_idx", outputCol="marital_onehot")

indexer_model = indexer.fit(train_df)
train_df = indexer_model.transform(train_df)
test_df = indexer_model.transform(test_df)

encoder_model = encoder.fit(train_df)
train_df = encoder_model.transform(train_df)
test_df = encoder_model.transform(test_df)

In [51]:
train_df.select("gender", "gender_encoded", "marital_status", "marital_onehot").show(3, truncate=False)

+------+--------------+--------------+--------------+
|gender|gender_encoded|marital_status|marital_onehot|
+------+--------------+--------------+--------------+
|F     |0             |S             |(4,[1],[1.0]) |
|F     |0             |S             |(4,[1],[1.0]) |
|F     |0             |M             |(4,[0],[1.0]) |
+------+--------------+--------------+--------------+
only showing top 3 rows



In [55]:
feature_cols = [
    "age",
    "total_encounters_before",
    "inpatient_count_before",
    "emergency_count_before",
    "avg_duration_before",
    "total_conditions_before",
    "chronic_conditions_before",
    "distinct_conditions_codes_before",
    "gender_encoded",
    "marital_onehot",
]
train_df.select(*feature_cols).show(3, truncate=False)

+---+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+--------------+
|age|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|gender_encoded|marital_onehot|
+---+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+--------------+
|42 |3                      |0                     |0                     |0.0                |5                      |5                        |5                               |0             |(4,[1],[1.0]) |
|58 |8                      |1                     |1                     |13.875             |7                      |7                        |7                  